# 08 Current collector (IA-COL-001)

**Requirement:** `IA-COL-001`

Patents: US12308414B2, EP4602674A1. Equations are paraphrased; see claims/paragraphs cited below.

## Governing equations


\(V_\mathrm{ohm} = I\,R(T,\mathrm{degradation})\). No fake lag on resistance.
Anode branch-plus-primary, ORR dual-face tabs, Ni-coated-steel + EPDM (US FIGS. 4B-4C),
SS mesh in iron ribs (EP), can-negative housing (US Claim 5).


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

here = Path.cwd()
for cand in (here, here.parent):
    if (cand / "src" / "ironair").is_dir() and (cand / "plant_sim").is_dir():
        sys.path.insert(0, str(cand))
        sys.path.insert(0, str(cand / "src"))
        break

from plant_sim.bootstrap import setup_path
setup_path()
from plant_sim.params import default_params, CAPACITY_FE_OH2_MAH_G, CAPACITY_MAGNETITE_MAH_G
from plant_sim.plotting import apply_style, timeseries, xy_plot, FIGURES
apply_style()
P = default_params()
print("6 M KOH default", P.c_KOH_mol_m3/1000, "M; T_ep", P.T_ep_sim_K, "K")
print("Faraday 960/320 checks", round(CAPACITY_FE_OH2_MAH_G,2), round(CAPACITY_MAGNETITE_MAH_G,2))
print("figures ->", FIGURES)


## Isolated component simulation

In [ ]:
from plant_sim.components.current_collector import CurrentCollector
from plant_sim.components.base import integrate_component
col = CurrentCollector(P)
t, y = integrate_component(lambda t,y: col.rhs(t,y,{"I_cell_A": 8.0, "T_amb_K": P.T_ep_sim_K}, {}), col.y0(), (0, 1500), n_eval=100)
V = [col.outputs(float(ti), y[:,i], {"I_cell_A": 8.0}, {})["V_ohm_V"] for i,ti in enumerate(t)]
timeseries(t/60, {"T": y[0], "deg": y[1], "Vohm": V}, xlabel="t (min)", ylabel="T / deg / V", title="Collector ohmic loss", callout="IA-COL-001", stem="nb08_col")


## Verification against patents / SSS002

In [ ]:
print('V_ohm = I * R(T, deg) with no extra lag state.')